# Python Setup

In [1]:
from sympy import *
from IPython.display import display, Math, Latex, Markdown
#init_session()
init_printing(use_unicode=True)

#init_printing(latex_printer=lambda *args, **kwargs: latex(*args, mul_symbol='dot', **kwargs))
def md(s):
    display(Markdown(s))
    
def note(s):
    md(f'<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">{s}</span></div>')

In [2]:
class dotdict(dict):
    __getattr__ = dict.get
    #__setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__
    
    def __setattr__(self, key, val):
        self.set(key, val)

    def set(self, key, val, extra_subs=None, unit=''):
        if not isinstance(val, str):
            return dict.__setitem__(self, key, val)
            
        ## Consider any string passe in as a SymPy expression
        expr = sympify(val, evaluate=False)

        with evaluate(False):
            expr_subs = expr.subs(self | (extra_subs or {}))
            subtituted_ltx = latex(expr_subs,mul_symbol='dot',order='lex')
        res = expr_subs.doit()
        eq = f'{key} = {latex(expr,mul_symbol='dot',order='lex')} = {subtituted_ltx} = {N(res, 5)} \\, {unit}'
        #print(eq)
        display(Math(eq))
        return dict.__setitem__(self, key, res)

# Geometrie

In [38]:
geom = dotdict({
    'd': 56,
    'l_h1': 5,
    'l_h2': 12.5,
    'h_a': 7,
    'stuetzweite': 6750,
})
geom.l_h = 'l_h1 + l_h2'

geom.α = atan(5/100)
geom.l = 'l_h/cos(α)'
geom.b = 'l'
geom.Δh = 'l_h * tan(α)'
geom.h = 'h_a + Δh'
geom.A = 'l * d'
#geom

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [71]:
#TODO choose material, rn I profil 15x15??

# HEB 240
obergurt = dotdict({
    'b': 240, #mm
    
})

# HEB 240
untergurt = dotdict({
    'b': 240, #mm
})

# HEB 160
stuetzeA = dotdict({
    'l': 6500, # mm
    'b': 160, #mm
    'd': 160, #mm
    'Iy': 2492e4, #mm^4
    'Iz': 889.2e4, #mm^4
    'A': 5425, #mm^2
})

# HEB 650
stuetzeB = dotdict({
    'l': 6250, # mm
    'b': 300, #mm
    'd': 650, #mm
    'Iy': 210600e4, #mm^4
    'Iz': 13980e4, #mm^4
    'A': 28600, # mm^2
})

binder = dotdict({
    'b': 240,
    'A_x': (5000 * 1500 * 0.5) + (12500 * 1000),
    'h': 1500,
})
pfette = dotdict({
    'l': geom.stuetzweite + binder.b,
    'e': geom.l/7,
    'abstand': geom.l/7,
})
pfette

{'l': 6990, 'e': 2.50312304931260, 'abstand': 2.50312304931260}

# Lastaufstellung

## Dachaufbau

In [5]:
dauerlasten = dotdict()
dauerlasten.set('g_wellblech', '0.4 * e/1000', pfette, unit='kN/m')

<IPython.core.display.Math object>

## Pfette

In [40]:
dauerlasten.set('g_pfette', '(7.75 + 0.12) * 0.423', pfette, unit='kN')
# Längste Pfette am Rand (mit kragarm): 7.75m
# Überlappung an linkem Binder: HEB240 * 0.5 => 0.12m
# last HEA 200: 0.423 kN/m lt https://www.ezzat.org/de/Querschnittswerte/gewalzt/I/hea.php

<IPython.core.display.Math object>

## Schnee

In [41]:
schnee = dotdict({ 'C_e': 1, 'C_t':  1, 'mu_i':  0.8, 's_k':  3.4, })

schnee.s = 'C_e * C_t * mu_i * s_k'
md('#### ✅ [11] Schneelast')
print(f'Schneelast s = {schnee.s:0.4} kN/m^2')

<IPython.core.display.Math object>

#### ✅ [11] Schneelast

Schneelast s = 2.720 kN/m^2


## Windlast

In [42]:
def init_wind():
    ## Werte lt hora.gv.at für "Bad Ischl (Stadt)"
    state = dotdict({
        'q_b0': 0.29,  # [kN/m^2]
        'v_b0': 21.50,  # [m/s] 
        'z_e': geom.h,  # [m]
        'cf_Binder': 2, #lt Angabe
    })
    
    ## Gelaendekategorie II
    state.q_p = 'q_b0 * 2.1 * (z_e/10)**0.24'
    return state
    

## Wind ⬆️ NORD (90°)

In [43]:
wind_nord = init_wind()
wind_nord

<IPython.core.display.Math object>

{'q_b0': 0.29,
 'v_b0': 21.5,
 'z_e': 7.87500000000000,
 'cf_Binder': 2,
 'q_p': 0.575065650879324}

In [44]:
## Reibung

wind_nord.c_fr = 0.04   # [EC3-1-4, Tab 7.10] "Sehr rauh" weil Trapezblech
wind_nord.set('A_fr', '2 * l * d', geom)
md('### ✅ [31] Reibung auf Dach')
wind_nord.F_fr_nord = 'c_fr * q_p * A_fr'

<IPython.core.display.Math object>

### ✅ [31] Reibung auf Dach

<IPython.core.display.Math object>

### Winddruck auf Tragstruktur

In [66]:
### Binder:
md('### ✅ [33] Windkraft auf FW Binder')
wind_nord.set('Fw_binder', 'cf_binder * q_p * (A_x)/10e6', binder)

### ✅ [33] Windkraft auf FW Binder

<IPython.core.display.Math object>

### ✅ [32] Windkraft auf Stützen

In [70]:
wind_nord.set('Fw_StuetzeA', 'cf_binder * q_p * (d * l)/1000**2', stuetzeA, unit='kN')
wind_nord.set('Fw_StuetzeB', 'cf_binder * q_p * (d * l)/1000**2', stuetzeB, unit='kN')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Wind ➡️ OST (0°)

In [26]:
wind_ost = init_wind()
wind_ost

<IPython.core.display.Math object>

{'q_b0': 0.29,
 'v_b0': 21.5,
 'z_e': 7.87500000000000,
 'cf_Binder': 2,
 'q_p': 0.575065650879324}

Es wirken:

 - Winddruck auf Dachfläche $w_k$
 - Reibung auf Dachfläche $w_r$
 - Winddruck auf Stütze (Reihe A)
 - Winddruck auf Wandriegel

In [27]:
def interp_alpha(val0, val5):
    return N(val0 + deg(geom.α) * ((val5-val0)/5),2)

#### Winddruck auf Dachfläche

In [47]:
## Flächeneinteilung 
md('**Dachflächen**')
geom.Dach_A = '(d - 2 * d/10) * (b - 2 * b/10)'
geom.Dach_B = '2 * (d * b/10)'
geom.Dach_C = '2 * (d/10 * b)'

# φ = 0 von dieser Richtung
cf_max = interp_alpha(0.2, 0.4)
cf_min = interp_alpha(-0.5, -0.7)
cpnet_maxA = interp_alpha(0.5, 0.8)
cpnet_maxB = interp_alpha(1.8, 2.1)
cpnet_maxC = interp_alpha(1.1, 1.3)

cpnet_minA = interp_alpha(-0.6, -1.1)
cpnet_minB = interp_alpha(-1.3, -1.7)
cpnet_minC = interp_alpha(-1.4, -1.8)

wind_ost.cpnet_min = sum([(geom.Dach_A/geom.A) * cpnet_minA,
                          (geom.Dach_B/geom.A) * cpnet_minB,
                          (geom.Dach_C/geom.A) * cpnet_minC])
wind_ost.cpnet_max = sum([(geom.Dach_A/geom.A) * cpnet_maxA,
                          (geom.Dach_B/geom.A) * cpnet_maxB,
                          (geom.Dach_C/geom.A) * cpnet_maxC])


md('### ✅ [21] Druck')
wind_ost.wk_1 = 'q_p * cpnet_max'

md('### ✅ [22] Sog')
wind_ost.wk_2 = 'q_p * cpnet_min'

**Dachflächen**

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### ✅ [21] Druck

<IPython.core.display.Math object>

### ✅ [22] Sog

<IPython.core.display.Math object>

### ✅ [23] Reibungskraft auf Dachfläche

In [53]:
# Da Trapezblech entlang windrichtung orientiert, kann ein $c_{fr} = 0.01$ angenommen werden
wind_ost.c_fr = 0.01
wind_ost.set('A_fr', '2 * l * d', geom)

wind_ost.w_fr = 'c_fr * q_p'
wind_ost.F_fr = 'c_fr * q_p * A_fr'

#randbinder:
wind_ost.set('F_frBinder', 'w_fr * l/1000 * stuetzweite/1000', geom, unit='kN')
wind_ost.F_fr * geom.stuetzweite


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### ✅ [24] Windkraft auf Stütze (Reihe A)

In [49]:
## Annahme: Im bereich C der kurve
wind_ost.set('cf0_col', '-0.721 * ln(d/b) + 2.1460', stuetzeA)
wind_ost.ψ_γ = 1 # kantiger QS
stuetzeA.λ = 'max(l/b, 70)'
note('effektive schlankheit passt so? [EC 1-4, Tab 7.16]')
wind_ost.ψ_λ = 1 #fuer φ = 0 und λ = 70 [EC 1-4, Abb. 7.36]
wind_ost.cf_col = 'cf0_col * ψ_λ'

wind_ost.set('F_frStuetze', 'q_p * cf_col * ((b*l)/1e6)', stuetzeA)



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">effektive schlankheit passt so? [EC 1-4, Tab 7.16]</span></div>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### ✅ [25] Windkraft auf Binder

In [62]:
wind_ost.set('F_frBinder', 'q_p * cf_Binder * (5154 * b)/1000**2', binder)

<IPython.core.display.Math object>

## Wind ⬅️ WEST (180°)

Es wirken:

 - Winddruck auf Dachfläche $w_k$
 - Reibung auf Dachfläche $w_r$
 - Winddruck auf Stütze (Reihe A)
 - Winddruck auf Wandriegel

In [58]:
wind_west = init_wind()
wind_west.set('φ', '(6+Δh)/h', geom)
wind_west

<IPython.core.display.Math object>

<IPython.core.display.Math object>

{'q_b0': 0.29,
 'v_b0': 21.5,
 'z_e': 7.87500000000000,
 'cf_Binder': 2,
 'q_p': 0.575065650879324,
 'φ': 0.873015873015873}

#### Winddruck auf Dachfläche

In [16]:
def interp_phi(val0, val1):
    return N(val0 + wind_west.φ * (val1-val0),2)

In [17]:
##### φ = elem (0,1) von dieser Richtung

cpnet_maxA = interp_alpha(0.5, 0.8)
cpnet_maxB = interp_alpha(1.8, 2.1)
cpnet_maxC = interp_alpha(1.1, 1.3)

cpnet_minA = interp_alpha(interp_phi(-0.6, -1.5), interp_phi(-1.1, -1.6))
cpnet_minB = interp_alpha(interp_phi(-1.3, -1.8), interp_phi(-1.7, -2.2))
cpnet_minC = interp_alpha(interp_phi(-1.4, -2.2), interp_phi(-1.8, -2.5))

wind_west.cpnet_min = sum([(geom.Dach_A/geom.A) * cpnet_minA,
                           (geom.Dach_B/geom.A) * cpnet_minB,
                           (geom.Dach_C/geom.A) * cpnet_minC])
wind_west.cpnet_max = sum([(geom.Dach_A/geom.A) * cpnet_maxA,
                           (geom.Dach_B/geom.A) * cpnet_maxB,
                           (geom.Dach_C/geom.A) * cpnet_maxC])


md('### ✅ [26] Druck')
wind_west.wk_1 = 'q_p * cpnet_max'

md('### ✅ [27] Sog')
wind_west.wk_2 = 'q_p * cpnet_min'

### ✅ [26] Druck

<IPython.core.display.Math object>

### ✅ [27] Sog

<IPython.core.display.Math object>

### ✅ [28] Reibungskraft auf Dachfläche

In [72]:
# Da Trapezblech entlang windrichtung orientiert, kann ein $c_{fr} = 0.01$ angenommen werden
wind_west.c_fr = 0.01
wind_west.set('A_fr', '2 * l * d', geom)
wind_west.F_fr = 'c_fr * q_p * A_fr'


<IPython.core.display.Math object>

<IPython.core.display.Math object>

### ✅ [29] Windkraft auf Stütze (Reihe B)

In [59]:
## Annahme: Im bereich C der kurve
wind_west.set('cf0_col', '-0.721 * ln(d/b) + 2.1460', stuetzeB)
#wind_west.ψ_γ = 1 # kantiger QS
#stuetzeB.λ = 'max(l/b, 70)'
#note('effektive schlankheit passt so? [EC 1-4, Tab 7.16]')
#wind_west.ψ_λ = 1 #fuer φ = 0 und λ = 70 [EC 1-4, Abb. 7.36]
#wind_west.cf_col = 'cf0_col * ψ_λ'
wind_west.set('F_frStuetze', 'q_p * cf_Binder * ((b*l)/1e6)', stuetzeB)



<IPython.core.display.Math object>

<IPython.core.display.Math object>

### ✅ [30] Windkraft FW-Binder

In [63]:
wind_west.set('F_frBinder', 'q_p * cf_Binder * ((11345 + 1469) * b)/1000**2', untergurt)

<IPython.core.display.Math object>

# Lastfälle

## Grundlastfälle

 - `GLF01` Eigengewicht der Stahlkonstruktion
 - `GLF02` Ausbaulasten (Trapezblech)
<hr style="width: 7rem; margin-left: 1rem;"/>
 - `GLF11` Schnee
<hr style="width: 7rem; margin-left: 1rem;"/>
 - `GLF21` Wind ➡️: Dach Druck
 - `GLF22` Wind ➡️: Dach Sog
 - `GLF23` Wind ➡️: Dach Reibung
 - `GLF24` Wind ➡️: Stuetzen
 - `GLF25` Wind ➡️: Wandriegel (??)
<hr style="width: 7rem; margin-left: 1rem;"/>
 - `GLF26` Wind ️️⬅️: Dach Druck
 - `GLF27` Wind ️️⬅️: Dach Sog
 - `GLF28` Wind ️️⬅️: Dach Reibung (??)
 - `GLF29` Wind ⬅️️: Stuetzen
 - `GLF30` Wind ⬅️: Wandriegel (??)
<hr style="width: 7rem; margin-left: 1rem;"/>
 - `GLF31` Wind ⬆️️️: Dach Reibung
 - `GLF32` Wind ⬆️️️: Stuetzen
 - `GLF33` Wind ⬆️️️: Binder
<hr style="width: 7rem; margin-left: 1rem;"/>
 - `GLF41` Nuztlast für Instandsetzungsarbeiten: Dach Kategorie H

## Einzellastfälle

| # | Bezeichnung | Formel |
|---|-------------|--------|
|`ELF10` | Eigengewichte, ständige Lasten | `GLF01 + GLF02`|
|`ELF20` | Schnee, veraenderlich | `GLF11`|
|`ELF40` | Wind ➡️ Druck | `GLF21 + GLF23 + GLF24 + GLF25`|
|`ELF41` | Wind ➡️ Sog   | `GLF22 + GLF23 + GLF24 + GLF25`|
|`ELF42` | Wind ⬅️ Druck | `GLF26 + GLF28 + GLF29 + GLF30`|
|`ELF43` | Wind ⬅️ Sog   | `GLF27 + GLF28 + GLF29 + GLF30`|
|`ELF44` | Wind ⬆️️       | `GLF31 + GLF32 + GLF33`        |
|`ELF50` | Nutzlast, veraenderlich | `GLF41`|

# Nachweise

## Stäbe mit Nachweisen

### (1) Pfettenstab

  - Einfeldträger
  - Gegen Kippen gehalten (Trapezblech)
  - **Einwirkung**: Einachsige Biegung & Druck
  - **Nachweise**:
    - [Biegung, Druck] Biege-knicken (um y)
    - [Biegung, Druck] Querschnitt

### (2) Dachverband Diagonale

  - Zugstab, Fachwerkstab
  - **Einwirkung**: Zug
  - **Nachweise**:
    - [Zug] Querschnitt

### (3) Binder: Obergurt

  - **Einwirkung**: Zug bzw Druck (Windsog)
  - **Nachweise**:
    - [Zug] Querschnit
    - [Druck] Knicken in Ebene
    - [Druck] Knicken Aus der Ebene
    - [Druck] Querschnitt
    - [Moment] Biegung

### (4) Binder: Untergurt

  - **Einwirkung**: Druck bzw Zug (Windsog)
  - **Nachweise**:
    - [Druck] Knicken in Ebene
    - [Druck] Knicken Aus der Ebene
    - [Druck] Querschnit
    - [Zug] Querschnitt
    - [Moment] Biegung

### (5) Binder: Füllstab

  - Finde maßgebliche Stäbe
  - **Einwirkung**: Druck bzw Zug
  - **Nachweise**:
    - [Druck] Knicken in Ebene
    - [Druck] Knicken Aus der Ebene
    - [Druck] ~Querschnit~ (implizit erfüllt)
    - [Zug] Querschnitt

### (6) Wandverband

  - **Einwirkung**: Zug
  - **Nachweise**:
    - [Zug] Querschnit

### (7) Stütze in Achse A

  - **Einwirkung**: Zug bzw Druck (Windsog)
  - **Nachweise**:
    - [Zug] Querschnit
    - [Druck] Biegeknicken aus Ebene
    - [Druck] Biegeknicken in Ebene

### (9) Wandriegel

  - **Einwirkung**: $\frac{N}{100}$ der Normalkraft in der Stütze A (bei LF Sog)
  - **Nachweise**:
    - [Zug] Querschnit
    - [Druck] Biegeknicken aus Ebene
    - [Druck] Biegeknicken in Ebene